# Optimization-seed variability

Subset selection is frozen at seed 42. Optimization seeds 42, 43 and 44 use exactly the same manifest. This notebook reports fresh experiments only.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

import pandas as pd
from IPython.display import display
from neyshekar_experiments.manifests import prepare
from neyshekar_experiments.training import experiment_grid, plan, execute
from neyshekar_experiments.reporting import family_scores, corrected_results, metric_figure

# Imports and reports never start training; paths use the shared repository root.

RUN_TRAINING = False
DATA_SEED = 42  # All manifests are frozen with this seed; optimization seeds are separate.

# Create/verify manifests from source data when this notebook is run.
manifest_summary = prepare()

## 1. Fixed training manifests

In [ ]:
import json
from neyshekar_experiments.manifests import load_manifest

display(manifest_summary[["name", "clips", "hours", "sha256"]])

## 2. WER and CER across completed seeds

Sample standard deviation describes training variability; it is not a confidence interval. Results remain empty until fresh runs finish.

In [ ]:
from neyshekar_experiments.reporting import seed_summary

display(seed_summary())
display(family_scores("matched"))

## 3. Corrected seed repetitions

In [ ]:
runs = experiment_grid("matched", seeds=(42, 43, 44))
display(plan(runs))
if RUN_TRAINING:
    execute(runs)

## 4. Sampling uncertainty

The reusable paired bootstrap computes both WER and CER. Speaker clustering requires a verified mapping; without it, the report says pending. Prompt clustering is a separate sensitivity analysis, not a replacement.

In [ ]:
from neyshekar_experiments.analysis import analyze_fresh

RUN_ANALYSIS = False
if RUN_ANALYSIS:
    display(analyze_fresh(replicates=20000))
else:
    print("Analysis disabled; enable after completing fresh decodings.")